# Notebook 03 - Data Cleaning and Preparation

## Objetivo

Este notebook tem como objetivo preparar a base final que será utilizada no dashboard.

As etapas incluem:

- Tratamento de valores ausentes
- Padronização de categorias
- Conversão da coluna de skills
- Tratamento da coluna salarial
- Investigação de outliers
- Criação do dataset final limpo


In [1]:
import pandas as pd
import numpy as np
import ast
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
df = pd.read_csv("../data/data_science_job_posts_2025.csv")

df.head()

,job_title,seniority_level,status,company,location,post_date,headquarter,industry,ownership,company_size,revenue,salary,skills
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea..."
1,data scientist,lead,hybrid,company_005,"Fort Worth, TX . Hybrid",15 days ago,"Detroit, MI, US",Manufacturing,Public,"155,030",€51.10B,"€118,733","['spark', 'r', 'python', 'sql', 'machine learn..."
2,data scientist,senior,on-site,company_007,"Austin, TX . Toronto, Ontario, Canada . Kirkla...",a month ago,"Redwood City, CA, US",Technology,Public,"25,930",€33.80B,"€94,987 - €159,559","['aws', 'git', 'python', 'docker', 'sql', 'mac..."
3,data scientist,senior,hybrid,company_008,"Chicago, IL . Scottsdale, AZ . Austin, TX . Hy...",8 days ago,"San Jose, CA, US",Technology,Public,"34,690",€81.71B,"€112,797 - €194,402","['sql', 'r', 'python']"
4,data scientist,NaN,on-site,company_009,On-site,3 days ago,"Stamford, CT, US",Finance,Private,"1,800",Private,"€114,172 - €228,337",[]


In [3]:
df_clean = df.copy()

print(f"Linhas: {df_clean.shape[0]}")
print(f"Colunas: {df_clean.shape[1]}")

Linhas: 944
Colunas: 13


In [4]:
missing = pd.DataFrame({
    "quantidade": df_clean.isnull().sum(),
    "percentual": (df_clean.isnull().mean() * 100).round(2)
})

missing.sort_values("percentual", ascending=False)

,quantidade,percentual
status,256,27.12
seniority_level,60,6.36
ownership,47,4.98
revenue,15,1.59
job_title,3,0.32
location,2,0.21
company,0,0.00
post_date,0,0.00
headquarter,0,0.00
industry,0,0.00


In [5]:
categorical_columns = [
    "status",
    "seniority_level",
    "ownership",
    "revenue",
    "job_title",
    "location",
    "company",
    "industry",
    "company_size"
]

for col in categorical_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Não informado")

In [6]:
df_clean.isnull().sum().sort_values(ascending=False)

job_title          0
seniority_level    0
status             0
company            0
location           0
post_date          0
headquarter        0
industry           0
ownership          0
company_size       0
revenue            0
salary             0
skills             0
dtype: int64

In [7]:
text_columns = [
    "job_title",
    "seniority_level",
    "status",
    "company",
    "location",
    "industry",
    "company_size",
    "ownership",
    "revenue"
]

for col in text_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip()

In [8]:
for col in ["job_title", "seniority_level", "status", "industry"]:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False))


job_title
job_title
data scientist               856
machine learning engineer     80
data engineer                  4
Não informado                  3
data analyst                   1
Name: count, dtype: int64

seniority_level
seniority_level
senior           630
lead             116
midlevel         113
Não informado     60
junior            25
Name: count, dtype: int64

status
status
on-site          363
Não informado    256
hybrid           207
remote           118
Name: count, dtype: int64

industry
industry
Technology       582
Finance          127
Retail           110
Healthcare        83
Education         19
Energy            12
Manufacturing      7
Logistics          4
Name: count, dtype: int64


In [9]:
def parse_skills(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [str(skill).strip().lower() for skill in value]

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                str(skill).strip().lower()
                for skill in parsed
                if str(skill).strip()
            ]
    except:
        pass

    return [
        skill.strip().lower()
        for skill in str(value).split(",")
        if skill.strip()
    ]

In [10]:
df_clean["skills_list"] = df_clean["skills"].apply(parse_skills)

df_clean[["skills", "skills_list"]].head()

,skills,skills_list
0,"['spark', 'r', 'python', 'scala', 'machine lea...","[spark, r, python, scala, machine learning, te..."
1,"['spark', 'r', 'python', 'sql', 'machine learn...","[spark, r, python, sql, machine learning]"
2,"['aws', 'git', 'python', 'docker', 'sql', 'mac...","[aws, git, python, docker, sql, machine learni..."
3,"['sql', 'r', 'python']","[sql, r, python]"
4,[],[]


In [11]:
df_clean["skills_count"] = df_clean["skills_list"].apply(len)

df_clean[["skills", "skills_list", "skills_count"]].head()

,skills,skills_list,skills_count
0,"['spark', 'r', 'python', 'scala', 'machine lea...","[spark, r, python, scala, machine learning, te...",6
1,"['spark', 'r', 'python', 'sql', 'machine learn...","[spark, r, python, sql, machine learning]",5
2,"['aws', 'git', 'python', 'docker', 'sql', 'mac...","[aws, git, python, docker, sql, machine learni...",9
3,"['sql', 'r', 'python']","[sql, r, python]",3
4,[],[],0


In [12]:
df_clean["skills_count"].describe()

count    944.000000
mean       4.429025
std        3.636046
min        0.000000
25%        1.000000
50%        4.000000
75%        7.000000
max       17.000000
Name: skills_count, dtype: float64

In [13]:
def extract_salary_values(value):
    if pd.isna(value):
        return pd.Series([np.nan, np.nan, np.nan])

    numbers = re.findall(r"\d[\d,]*", str(value))

    numbers = [
        int(num.replace(",", ""))
        for num in numbers
    ]

    if len(numbers) == 0:
        return pd.Series([np.nan, np.nan, np.nan])

    if len(numbers) == 1:
        salary_min = numbers[0]
        salary_max = numbers[0]
    else:
        salary_min = min(numbers)
        salary_max = max(numbers)

    salary_avg = (salary_min + salary_max) / 2

    return pd.Series([salary_min, salary_max, salary_avg])

In [14]:
df_clean[
    ["salary_min", "salary_max", "salary_avg"]
] = df_clean["salary"].apply(extract_salary_values)

df_clean[
    ["salary", "salary_min", "salary_max", "salary_avg"]
].head(20)

,salary,salary_min,salary_max,salary_avg
0,"€100,472 - €200,938",100472.0,200938.0,150705.0
1,"€118,733",118733.0,118733.0,118733.0
2,"€94,987 - €159,559",94987.0,159559.0,127273.0
3,"€112,797 - €194,402",112797.0,194402.0,153599.5
4,"€114,172 - €228,337",114172.0,228337.0,171254.5
5,"€196,371 - €251,170",196371.0,251170.0,223770.5
6,"€51,330 - €70,144",51330.0,70144.0,60737.0
7,"€121,480 - €132,440",121480.0,132440.0,126960.0
8,"€207,331",207331.0,207331.0,207331.0
9,"€219,201",219201.0,219201.0,219201.0


In [15]:
df_clean[["salary_min", "salary_max", "salary_avg"]].describe()

,salary_min,salary_max,salary_avg
count,9.440000e+02,9.440000e+02,9.440000e+02
mean,1.145134e+05,1.490456e+05,1.317795e+05
std,1.262089e+05,1.341397e+05,1.288144e+05
min,4.055000e+03,7.678000e+03,7.055000e+03
25%,6.828550e+04,8.237000e+04,7.637175e+04
50%,1.104585e+05,1.514690e+05,1.347240e+05
75%,1.461402e+05,2.009362e+05,1.697330e+05
max,2.739979e+06,2.739979e+06,2.739979e+06


In [16]:
df_clean[df_clean["salary_avg"] > 500000][
    [
        "job_title",
        "seniority_level",
        "company",
        "industry",
        "salary",
        "salary_min",
        "salary_max",
        "salary_avg"
    ]
]

,job_title,seniority_level,company,industry,salary,salary_min,salary_max,salary_avg
108,data scientist,senior,company_209,Technology,"€639,348",639348.0,639348.0,639348.0
536,data scientist,senior,company_967,Technology,"€2,739,979",2739979.0,2739979.0,2739979.0
702,data scientist,senior,company_967,Technology,"€2,283,322",2283322.0,2283322.0,2283322.0


In [17]:
df_clean["salary_outlier"] = df_clean["salary_avg"] > 500000

df_clean["salary_outlier"].value_counts()

salary_outlier
False    941
True       3
Name: count, dtype: int64

In [18]:
df_clean["salary_avg_adjusted"] = df_clean["salary_avg"]

df_clean.loc[
    df_clean["salary_outlier"] == True,
    "salary_avg_adjusted"
] = np.nan

df_clean[["salary_avg", "salary_avg_adjusted", "salary_outlier"]].describe()

,salary_avg,salary_avg_adjusted
count,9.440000e+02,941.000000
mean,1.317795e+05,126181.933581
std,1.288144e+05,64646.339484
min,7.055000e+03,7055.000000
25%,7.637175e+04,76371.000000
50%,1.347240e+05,134721.500000
75%,1.697330e+05,169155.500000
max,2.739979e+06,306626.000000


In [19]:
df_clean["salary_avg_adjusted"].describe()

count       941.000000
mean     126181.933581
std       64646.339484
min        7055.000000
25%       76371.000000
50%      134721.500000
75%      169155.500000
max      306626.000000
Name: salary_avg_adjusted, dtype: float64

In [20]:
df_skills_clean = (
    df_clean
    .explode("skills_list")
    .rename(columns={"skills_list": "skill"})
)

df_skills_clean = df_skills_clean[
    df_skills_clean["skill"].notna()
]

df_skills_clean = df_skills_clean[
    df_skills_clean["skill"] != ""
]

df_skills_clean.head()

,job_title,seniority_level,status,company,location,post_date,headquarter,industry,ownership,company_size,revenue,salary,skills,skill,skills_count,salary_min,salary_max,salary_avg,salary_outlier,salary_avg_adjusted
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea...",spark,6,100472.0,200938.0,150705.0,False,150705.0
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea...",r,6,100472.0,200938.0,150705.0,False,150705.0
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea...",python,6,100472.0,200938.0,150705.0,False,150705.0
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea...",scala,6,100472.0,200938.0,150705.0,False,150705.0
0,data scientist,senior,hybrid,company_003,"Grapevine, TX . Hybrid",17 days ago,"Bentonville, AR, US",Retail,Public,€352.44B,Public,"€100,472 - €200,938","['spark', 'r', 'python', 'scala', 'machine lea...",machine learning,6,100472.0,200938.0,150705.0,False,150705.0


In [21]:
df_skills_clean["skill"].value_counts().head(20)

skill
python              640
machine learning    580
sql                 442
r                   343
aws                 218
deep learning       178
tensorflow          165
spark               161
azure               155
pytorch             148
tableau             116
gcp                 106
scikit-learn         91
scala                85
database             83
pandas               76
java                 73
hadoop               67
git                  65
numpy                60
Name: count, dtype: int64

In [22]:
df_clean["is_senior_or_lead"] = df_clean["seniority_level"].isin(
    ["senior", "lead"]
)

df_clean["is_remote"] = df_clean["status"].eq("remote")

df_clean["is_data_scientist"] = df_clean["job_title"].eq("data scientist")

df_clean[[
    "seniority_level",
    "status",
    "job_title",
    "is_senior_or_lead",
    "is_remote",
    "is_data_scientist"
]].head()

,seniority_level,status,job_title,is_senior_or_lead,is_remote,is_data_scientist
0,senior,hybrid,data scientist,True,False,True
1,lead,hybrid,data scientist,True,False,True
2,senior,on-site,data scientist,True,False,True
3,senior,hybrid,data scientist,True,False,True
4,Não informado,on-site,data scientist,False,False,True


In [23]:
print(f"Linhas finais: {df_clean.shape[0]}")
print(f"Colunas finais: {df_clean.shape[1]}")
print(
    f"Duplicados: {df_clean.drop(columns=['skills_list']).duplicated().sum()}"
)

df_clean.info()

Linhas finais: 944
Colunas finais: 23
Duplicados: 0
<class 'pandas.DataFrame'>
RangeIndex: 944 entries, 0 to 943
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_title            944 non-null    str    
 1   seniority_level      944 non-null    str    
 2   status               944 non-null    str    
 3   company              944 non-null    str    
 4   location             944 non-null    str    
 5   post_date            944 non-null    str    
 6   headquarter          944 non-null    str    
 7   industry             944 non-null    str    
 8   ownership            944 non-null    str    
 9   company_size         944 non-null    str    
 10  revenue              944 non-null    str    
 11  salary               944 non-null    str    
 12  skills               944 non-null    str    
 13  skills_list          944 non-null    object 
 14  skills_count         944 non-null    int64  
 15 

In [24]:
print("Resumo do dataset preparado:")
print(f"Total de vagas: {len(df_clean)}")
print(f"Total de empresas: {df_clean['company'].nunique()}")
print(f"Total de setores: {df_clean['industry'].nunique()}")
print(f"Total de skills únicas: {df_skills_clean['skill'].nunique()}")
print(f"Salário médio ajustado: €{df_clean['salary_avg_adjusted'].mean():,.2f}")
print(f"Vagas Senior ou Lead: {df_clean['is_senior_or_lead'].mean() * 100:.2f}%")
print(f"Vagas remotas: {df_clean['is_remote'].mean() * 100:.2f}%")

Resumo do dataset preparado:
Total de vagas: 944
Total de empresas: 420
Total de setores: 8
Total de skills únicas: 33
Salário médio ajustado: €126,181.93
Vagas Senior ou Lead: 79.03%
Vagas remotas: 12.50%


In [25]:
df_clean.to_csv(
    "../data/data_science_jobs_clean.csv",
    index=False
)

print("Dataset limpo salvo com sucesso.")

Dataset limpo salvo com sucesso.


In [26]:
df_skills_clean.to_csv(
    "../data/data_science_jobs_skills_clean.csv",
    index=False
)

print("Dataset de skills salvo com sucesso.")

Dataset de skills salvo com sucesso.


In [27]:
df_check = pd.read_csv("../data/data_science_jobs_clean.csv")
df_skills_check = pd.read_csv("../data/data_science_jobs_skills_clean.csv")

print(df_check.shape)
print(df_skills_check.shape)

(944, 23)
(4181, 20)


In [28]:
print("""
Conclusão da preparação dos dados:

1. Valores ausentes categóricos foram preenchidos com 'Não informado'.
2. Textos categóricos foram padronizados.
3. A coluna de skills foi convertida em lista.
4. Foi criada uma base explodida para análise individual de skills.
5. A coluna de salário foi convertida em salary_min, salary_max e salary_avg.
6. Outliers salariais extremos foram identificados.
7. Foi criada uma coluna salary_avg_adjusted para análises salariais no dashboard.
8. Os datasets finais foram salvos na pasta data.
""")


Conclusão da preparação dos dados:

1. Valores ausentes categóricos foram preenchidos com 'Não informado'.
2. Textos categóricos foram padronizados.
3. A coluna de skills foi convertida em lista.
4. Foi criada uma base explodida para análise individual de skills.
5. A coluna de salário foi convertida em salary_min, salary_max e salary_avg.
6. Outliers salariais extremos foram identificados.
7. Foi criada uma coluna salary_avg_adjusted para análises salariais no dashboard.
8. Os datasets finais foram salvos na pasta data.

